In [12]:
#Importing libraries

from keras.preprocessing import text
from keras.utils import np_utils
from keras.preprocessing import sequence
from keras.utils import pad_sequences
import numpy as np
import pandas as pd


In [26]:
#Taking random sentences as data
data = """Deep learning (also known as deep structured learning) is part of a broader family
of machine learning methods based on artificial neural networks with representation
learning. Learning can be supervised, semi-supervised or unsupervised.
Deep-learning architectures such as deep neural networks, deep belief networks, deep
reinforcement learning, recurrent neural networks, convolutional neural networks and
Transformers have been applied to fields including computer vision, speech recognition,
natural language processing, machine translation, bioinformatics, drug design, medical
image analysis, climate science, material inspection and board game programs, where
they have produced results comparable to and in some cases surpassing human expert 
 performance.
"""
dl_data = data.split()

In [27]:
#a. Data preparation
#Tokenization
tokenizer = text.Tokenizer()
tokenizer.fit_on_texts(dl_data)
word2id = tokenizer.word_index
word2id['PAD'] = 0
id2word = {v:k for k, v in word2id.items()}
wids = [[word2id[w] for w in text.text_to_word_sequence(doc)] for doc in dl_data]
vocab_size = len(word2id)
embed_size = 100
window_size = 2
print('Vocabulary Size:', vocab_size)
print('Vocabulary Sample:', list(word2id.items())[:10])


Vocabulary Size: 75
Vocabulary Sample: [('learning', 1), ('deep', 2), ('networks', 3), ('neural', 4), ('and', 5), ('as', 6), ('of', 7), ('machine', 8), ('supervised', 9), ('have', 10)]


In [28]:
#b. Generate training data
#Generating (context word, target/label word) pairs
def generate_context_word_pairs(corpus, window_size, vocab_size):
    context_length = window_size*2
    for words in corpus:
        sentence_length = len(words)
        for index, word in enumerate(words):
            context_words = []
            label_word = []
            start = index - window_size
            end = index + window_size + 1
            context_words.append([words[i]
                for i in range(start, end)
                if 0 <= i < sentence_length and i != index])
            label_word.append(word)
            x = pad_sequences(context_words, maxlen=context_length)
            y = np_utils.to_categorical(label_word, vocab_size)
            yield (x, y)
i = 0
for x, y in generate_context_word_pairs(corpus=wids, window_size=window_size,vocab_size=vocab_size):
    if 0 not in x[0]:

        print('Context (X):', [id2word[w] for w in x[0]], '-> Target (Y):',id2word[np.argwhere(y[0])[0][0]])
        if i == 10:
            break
        i += 1


In [29]:
#c. Train model
#Model building
import keras.backend as K
from keras.models import Sequential
from keras.layers import Dense, Embedding, Lambda
cbow = Sequential()
cbow.add(Embedding(input_dim=vocab_size, output_dim=embed_size,input_length=window_size*2))
cbow.add(Lambda(lambda x: K.mean(x, axis=1), output_shape=(embed_size,)))
cbow.add(Dense(vocab_size, activation='softmax'))
cbow.compile(loss='categorical_crossentropy', optimizer='rmsprop')
print(cbow.summary())

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_1 (Embedding)     (None, 4, 100)            7500      
                                                                 
 lambda_1 (Lambda)           (None, 100)               0         
                                                                 
 dense_1 (Dense)             (None, 75)                7575      
                                                                 
Total params: 15,075
Trainable params: 15,075
Non-trainable params: 0
_________________________________________________________________
None


In [30]:
for epoch in range(1, 6):
    loss = 0.
    i = 0
    for x, y in generate_context_word_pairs(corpus=wids,window_size=window_size, vocab_size=vocab_size):
        i += 1
        loss += cbow.train_on_batch(x, y)
        if i % 100000 == 0:
            print('Processed {} (context, word) pairs'.format(i))
    print('Epoch:', epoch, '\tLoss:', loss)
    print()

Epoch: 1 	Loss: 433.6151604652405

Epoch: 2 	Loss: 429.1187252998352

Epoch: 3 	Loss: 425.899489402771

Epoch: 4 	Loss: 422.8608741760254

Epoch: 5 	Loss: 420.4263834953308



In [31]:
weights = cbow.get_weights()[0]
weights = weights[1:]
print(weights.shape)
pd.DataFrame(weights, index=list(id2word.values())[1:]).head()

(74, 100)


,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
deep,0.025086,-0.052213,0.007580,0.040589,-0.058755,-0.003015,0.042260,0.020879,-0.030195,-0.018637,...,0.038087,-0.048115,-0.047540,-0.006379,0.003884,-0.007135,0.032407,0.031271,-0.014645,0.037700
networks,-0.004432,0.032190,0.016236,0.038939,-0.057518,-0.022406,0.010631,0.029116,-0.061280,-0.030781,...,0.002875,0.006836,-0.003318,-0.029295,-0.062762,0.015358,0.005709,0.026443,-0.022353,0.030095
neural,0.020533,0.023668,0.047101,0.030435,-0.007649,-0.046225,-0.020052,-0.015875,-0.038146,0.030969,...,0.031244,0.007732,-0.013097,-0.038053,0.038101,0.028604,-0.018818,-0.008720,0.024229,-0.040129
and,-0.030833,0.033455,-0.047359,-0.043407,-0.041254,-0.043584,0.009719,-0.028110,-0.003623,0.000578,...,-0.048338,0.022541,0.019452,-0.018393,-0.011670,-0.029422,0.005820,-0.008711,-0.017067,0.048606
as,0.002398,-0.046294,-0.014023,0.038871,0.017468,0.025368,-0.013243,-0.004607,0.034917,-0.008929,...,0.009863,-0.043584,-0.040368,0.049707,-0.044258,-0.036769,0.016917,0.022594,0.015265,0.028806


In [32]:
#d. Output
from sklearn.metrics.pairwise import euclidean_distances
distance_matrix = euclidean_distances(weights)
print(distance_matrix.shape)
similar_words = {search_term: [id2word[idx] for idx in distance_matrix[word2id[search_term]-1].argsort()[1:6]+1] for
search_term in ['deep']}
similar_words


(74, 74)


{'deep': ['of', 'where', 'a', 'bioinformatics', 'translation']}